# 🧹 Notebook 02 — Data Preprocessing
**Healthcare RAG-Powered Medical Q&A Assistant**

**Goal:** Take the raw CSV and turn it into a clean, ready-to-use dataset.

The raw data has 3 columns:
- `instruction` → contains the medical context (long paragraph)
- `input` → contains the question
- `output` → contains the answer

We will clean these and save a new file called `pubmedqa_cleaned.csv`

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import re
import os

print('Libraries imported successfully!')

## Step 2 — Load the Raw Data

In [ ]:
# Load the raw dataset
df = pd.read_csv('../data/raw/pubmedqa_raw.csv')

print('Number of rows:', len(df))
print('Columns:', df.columns.tolist())
print()

# Preview the first row
df.head(2)

## Step 3 — Extract Question, Context, and Answer

The raw columns have extra text we don't need:
- `instruction` starts with *"Answer the question based on the following context:"* — we only want the context part
- `input` starts with *"Question:"* — we only want the question text
- `output` is already clean — we rename it to `answer`

In [ ]:
# Extract only the context text from the instruction column
# Example: "Answer the question based on the following context: <THIS PART>"
def get_context(text):
    match = re.search(r'context:\s*(.*)', text, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()

# Extract only the question text from the input column
# Example: "Question: <THIS PART>"
def get_question(text):
    match = re.search(r'Question:\s*(.*)', text, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()

# Apply the functions
df['question'] = df['input'].apply(get_question)
df['context']  = df['instruction'].apply(get_context)
df['answer']   = df['output']

# Keep only the 3 new clean columns
df = df[['question', 'context', 'answer']]

print('Done! Here is a sample:')
print()
print('QUESTION:', df['question'][0])
print()
print('CONTEXT (first 200 chars):', df['context'][0][:200])
print()
print('ANSWER:', df['answer'][0])

## Step 4 — Clean the Text

We remove things that are not useful for NLP:
- Extra spaces
- Special characters like HTML tags
- Very long spaces or newlines

In [ ]:
def clean_text(text):
    # Remove extra spaces and newlines
    text = re.sub(r'\s+', ' ', str(text))
    # Remove HTML tags if any
    text = re.sub(r'<[^>]+>', '', text)
    # Remove leading and trailing spaces
    text = text.strip()
    return text

df['question'] = df['question'].apply(clean_text)
df['context']  = df['context'].apply(clean_text)
df['answer']   = df['answer'].apply(clean_text)

print('Text cleaned!')
print('Sample cleaned question:', df['question'][0])

## Step 5 — Check for Missing Values

In [ ]:
print('Missing values in each column:')
print(df.isnull().sum())
print()

# Remove any row that has no question or no answer
before = len(df)
df = df.dropna(subset=['question', 'answer'])
df = df.reset_index(drop=True)
after = len(df)

print(f'Removed {before - after} rows with missing values')
print(f'Remaining rows: {after:,}')

## Step 6 — Remove Duplicate Questions

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=['question'])
df = df.reset_index(drop=True)
after = len(df)

print(f'Removed {before - after} duplicate questions')
print(f'Remaining rows: {after:,}')

## Step 7 — Label Each Question with a Medical Category

We assign a category to each question by checking if certain keywords appear in the question text.

Categories: **Symptoms, Diagnosis, Treatment, Medication, Prevention, General**

In [ ]:
# Keywords for each category
# If a question contains any of these words → assign that category
categories = {
    'Symptoms':   ['symptom', 'sign', 'pain', 'fever', 'nausea', 'bleed',
                   'fatigue', 'vomit', 'diarrhea', 'swelling', 'rash'],

    'Diagnosis':  ['diagnos', 'detect', 'screen', 'test', 'scan', 'biopsy',
                   'assess', 'predict', 'marker', 'accuracy'],

    'Treatment':  ['treat', 'therapy', 'cure', 'surgery', 'operation',
                   'procedure', 'manage', 'intervention', 'radiation'],

    'Medication': ['drug', 'medication', 'dose', 'antibiotic', 'vaccine',
                   'pill', 'inject', 'supplement', 'steroid', 'inhibitor'],

    'Prevention': ['prevent', 'protect', 'avoid', 'lifestyle',
                   'diet', 'exercise', 'smoking', 'obesity']
}

def assign_category(question):
    question_lower = question.lower()
    for category, keywords in categories.items():
        for keyword in keywords:
            if keyword in question_lower:
                return category
    return 'General'  # default if no keyword matches

df['category'] = df['question'].apply(assign_category)

print('Category distribution:')
print(df['category'].value_counts())

## Step 8 — Add Useful Length Columns

These will help us during EDA to understand the data better.

In [ ]:
# Count number of words in each column
df['question_word_count'] = df['question'].str.split().str.len()
df['answer_word_count']   = df['answer'].str.split().str.len()
df['context_word_count']  = df['context'].str.split().str.len()

print('New columns added!')
print()
print('Average question length (words):', df['question_word_count'].mean().round(1))
print('Average answer length   (words):', df['answer_word_count'].mean().round(1))
print('Average context length  (words):', df['context_word_count'].mean().round(1))

## Step 9 — Final Check & Save

In [ ]:
# Final summary
print('===== FINAL DATASET SUMMARY =====')
print(f'Total rows     : {len(df):,}')
print(f'Total columns  : {len(df.columns)}')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Categories     : {df["category"].nunique()}')
print(f'Columns        : {df.columns.tolist()}')
print('=================================')

# Save the cleaned file
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/pubmedqa_cleaned.csv', index=False)

print()
print('Saved to: data/processed/pubmedqa_cleaned.csv')

## ✅ Done!

| Step | What we did |
|---|---|
| Extract | Got clean question, context, answer from raw columns |
| Clean | Removed extra spaces and HTML |
| Missing | Removed rows with no question or answer |
| Duplicates | Removed repeated questions |
| Categories | Labeled each question with a medical category |
| Features | Added word count columns |

**➡️ Next: Open `03_eda.ipynb`**